# Module 2.1 — Problem Framing, Label Schema & Data Acquisition
**DeskMate SLM Curriculum · Phase 2 · Notebook 08**

Run on **Google Colab** or **Kaggle** (CPU is fine — no GPU needed).

By the end you will have:
- DeskMate's unified intent/category/priority schema applied to two public datasets
- A frozen gold evaluation set (`data/gold/`) never used in training
- Leak-free train/val/test splits saved to `data/processed/`
- A printed datasheet showing class distributions per split

Read `2.1_data_acquisition.md` for the full theory and design decisions.

---


## Step 0 — Install Dependencies

In [ ]:
%%capture
!pip install -q datasets==2.21.0 sentence-transformers==3.1.1 pandas==2.2.2 scikit-learn==1.5.1


In [ ]:
import random, json, hashlib, pathlib
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Libraries loaded.")


## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_RAW  = PROJECT_ROOT / "data" / "raw"
DATA_PROC = PROJECT_ROOT / "data" / "processed"
DATA_GOLD = PROJECT_ROOT / "data" / "gold"
for p in [DATA_RAW, DATA_PROC, DATA_GOLD]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Runtime : {RUNTIME}")
print(f"Data dir: {PROJECT_ROOT / 'data'}")


## Step 2 — DeskMate Label Schema

In [ ]:
# Intent taxonomy — 14 classes + out_of_scope
INTENTS = [
    "account_access",
    "account_settings",
    "billing_dispute",
    "billing_inquiry",
    "cancellation",
    "data_privacy",
    "feature_request",
    "onboarding",
    "outage_report",
    "payment_failure",
    "performance_issue",
    "refund_request",
    "technical_bug",
    "usage_question",
    "out_of_scope",
]

# Category mapping
INTENT_TO_CATEGORY = {
    "account_access":   "account",
    "account_settings": "account",
    "billing_dispute":  "billing",
    "billing_inquiry":  "billing",
    "cancellation":     "billing",
    "data_privacy":     "data",
    "feature_request":  "product",
    "onboarding":       "product",
    "outage_report":    "technical",
    "payment_failure":  "billing",
    "performance_issue":"technical",
    "refund_request":   "billing",
    "technical_bug":    "technical",
    "usage_question":   "product",
    "out_of_scope":     "out_of_scope",
}

# Priority inference by intent (default rules — overrideable by keywords)
INTENT_TO_PRIORITY = {
    "account_access":   "medium",
    "account_settings": "low",
    "billing_dispute":  "high",
    "billing_inquiry":  "low",
    "cancellation":     "medium",
    "data_privacy":     "high",
    "feature_request":  "low",
    "onboarding":       "low",
    "outage_report":    "high",
    "payment_failure":  "high",
    "performance_issue":"medium",
    "refund_request":   "medium",
    "technical_bug":    "medium",
    "usage_question":   "low",
    "out_of_scope":     "low",
}

print(f"Intents   : {len(INTENTS)} (including out_of_scope)")
print(f"Categories: {sorted(set(INTENT_TO_CATEGORY.values()))}")
print(f"Priorities: {sorted(set(INTENT_TO_PRIORITY.values()))}")


## Step 3 — Load Public Datasets

We load two public datasets from HuggingFace Hub and inspect their schemas
before applying the DeskMate label mapping.


In [ ]:
print("Loading PolyAI/banking77 ...")
banking = load_dataset("PolyAI/banking77", trust_remote_code=True)
print(f"  Splits: {list(banking.keys())}")
print(f"  Train : {len(banking['train'])} examples")
print(f"  Test  : {len(banking['test'])} examples")
print(f"  Features: {banking['train'].features}")
print(f"  Sample: {banking['train'][0]}")


In [ ]:
print("Loading bitext customer support dataset ...")
bitext = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", trust_remote_code=True)
print(f"  Splits   : {list(bitext.keys())}")
print(f"  Train    : {len(bitext['train'])} examples")
print(f"  Features : {list(bitext['train'].features.keys())}")
print(f"  Sample   : {dict(list(bitext['train'][0].items())[:4])}")


In [ ]:
# Show class distributions
print("banking77 — top 10 intents:")
b77_labels = banking['train'].features['label'].names
b77_counts = Counter(banking['train']['label'])
for idx, cnt in b77_counts.most_common(10):
    print(f"  {b77_labels[idx]:<45} {cnt:>5}")

print()
print("bitext — intent distribution:")
bx_counts = Counter(bitext['train']['intent'])
for intent, cnt in sorted(bx_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {intent:<45} {cnt:>5}")


## Step 4 — Intent Mapping

Map source-dataset intents to DeskMate's 15-class schema.
Unmappable intents → `out_of_scope`.


In [ ]:
# banking77 → DeskMate mapping (banking domain, so many map to billing/account)
B77_MAP = {
    "activate_my_card":                      "account_settings",
    "age_limit":                             "usage_question",
    "apple_pay_or_google_pay":               "usage_question",
    "atm_support":                           "usage_question",
    "automatic_top_up":                      "account_settings",
    "balance_not_updated_after_bank_transfer":"billing_dispute",
    "balance_not_updated_after_cheque_or_cash_deposit": "billing_dispute",
    "beneficiary_not_allowed":               "technical_bug",
    "cancel_transfer":                       "cancellation",
    "card_about_to_expire":                  "account_settings",
    "card_acceptance":                       "technical_bug",
    "card_arrival":                          "billing_inquiry",
    "card_delivery_estimate":                "billing_inquiry",
    "card_linking":                          "technical_bug",
    "card_not_working":                      "payment_failure",
    "card_payment_fee_charged":              "billing_dispute",
    "card_payment_not_recognised":           "billing_dispute",
    "card_payment_wrong_exchange_rate":      "billing_dispute",
    "card_swallowed":                        "technical_bug",
    "cash_withdrawal_charge":                "billing_dispute",
    "cash_withdrawal_not_recognised":        "billing_dispute",
    "change_pin":                            "account_settings",
    "compromised_card":                      "data_privacy",
    "contactless_not_working":               "technical_bug",
    "country_support":                       "usage_question",
    "declined_card_payment":                 "payment_failure",
    "declined_cash_withdrawal":              "payment_failure",
    "declined_transfer":                     "payment_failure",
    "direct_debit_payment_not_recognised":   "billing_dispute",
    "disposable_card_limits":                "usage_question",
    "edit_personal_details":                 "account_settings",
    "exchange_charge":                       "billing_inquiry",
    "exchange_rate":                         "billing_inquiry",
    "exchange_via_app":                      "usage_question",
    "extra_charge_on_statement":             "billing_dispute",
    "failed_transfer":                       "technical_bug",
    "fiat_currency_support":                 "usage_question",
    "get_disposable_virtual_card":           "usage_question",
    "get_physical_card":                     "usage_question",
    "getting_spare_card":                    "usage_question",
    "getting_virtual_card":                  "usage_question",
    "lost_or_stolen_card":                   "data_privacy",
    "lost_or_stolen_phone":                  "data_privacy",
    "order_physical_card":                   "usage_question",
    "passcode_forgotten":                    "account_access",
    "pending_card_payment":                  "billing_inquiry",
    "pending_cash_withdrawal":               "billing_inquiry",
    "pending_top_up":                        "billing_inquiry",
    "pending_transfer":                      "billing_inquiry",
    "pin_blocked":                           "account_access",
    "receiving_money":                       "billing_inquiry",
    "refund_not_showing_up":                 "refund_request",
    "request_refund":                        "refund_request",
    "reverted_card_payment?":                "billing_dispute",
    "supported_cards_and_currencies":        "usage_question",
    "terminate_account":                     "cancellation",
    "top_up_by_bank_transfer_charge":        "billing_dispute",
    "top_up_by_card_charge":                 "billing_dispute",
    "top_up_by_cash_or_cheque":              "usage_question",
    "top_up_failed":                         "payment_failure",
    "top_up_limits":                         "usage_question",
    "top_up_reverted":                       "billing_dispute",
    "topping_up_by_card":                    "usage_question",
    "transaction_charged_twice":             "billing_dispute",
    "transfer_fee_charged":                  "billing_dispute",
    "transfer_into_account":                 "usage_question",
    "transfer_not_received_by_recipient":    "technical_bug",
    "transfer_timing":                       "billing_inquiry",
    "unable_to_verify_identity":             "account_access",
    "verify_my_identity":                    "account_access",
    "verify_source_of_funds":                "account_access",
    "verify_top_up":                         "billing_inquiry",
    "virtual_card_not_working":              "technical_bug",
    "visa_or_mastercard":                    "usage_question",
    "why_verify_identity":                   "usage_question",
    "wrong_amount_of_cash_received":         "billing_dispute",
    "wrong_exchange_rate_for_cash_withdrawal":"billing_dispute",
}

print(f"banking77 intents total    : {len(b77_labels)}")
print(f"Mapped to DeskMate intents : {len(set(B77_MAP.values()))}")
print(f"Unmapped (→ out_of_scope)  : {len(b77_labels) - len(B77_MAP)}")


In [ ]:
# bitext → DeskMate mapping
BITEXT_MAP = {
    "cancel_order":         "cancellation",
    "change_order":         "account_settings",
    "change_shipping_address": "account_settings",
    "check_cancellation_fee":  "billing_inquiry",
    "check_invoices":          "billing_inquiry",
    "check_payment_methods":   "billing_inquiry",
    "check_refund_policy":     "billing_inquiry",
    "complaint":               "billing_dispute",
    "contact_customer_service":"usage_question",
    "contact_human_agent":     "usage_question",
    "create_account":          "onboarding",
    "delete_account":          "cancellation",
    "delivery_options":        "usage_question",
    "delivery_period":         "usage_question",
    "edit_account":            "account_settings",
    "get_invoice":             "billing_inquiry",
    "get_refund":              "refund_request",
    "newsletter_subscription": "account_settings",
    "payment_issue":           "payment_failure",
    "place_order":             "usage_question",
    "recover_password":        "account_access",
    "registration_problems":   "account_access",
    "review":                  "feature_request",
    "set_up_shipping_address": "account_settings",
    "switch_account":          "account_settings",
    "track_order":             "usage_question",
    "track_refund":            "refund_request",
}

print(f"bitext intents total       : {len(set(bitext['train']['intent']))}")
print(f"Mapped to DeskMate intents : {len(set(BITEXT_MAP.values()))}")


## Step 5 — Build Unified Dataset

In [ ]:
def make_example(text, intent, source):
    category = INTENT_TO_CATEGORY.get(intent, "out_of_scope")
    priority  = INTENT_TO_PRIORITY.get(intent, "low")
    # Upgrade priority for high-signal keywords in text
    text_lower = text.lower()
    if any(w in text_lower for w in ["urgent", "emergency", "immediately", "data breach",
                                      "hacked", "stolen", "service down", "outage",
                                      "charged twice", "double charged", "cannot access"]):
        if priority == "low":
            priority = "medium"
        elif priority == "medium":
            priority = "high"
    return {
        "text":     text.strip(),
        "intent":   intent,
        "category": category,
        "priority": priority,
        "source":   source,
    }

rows = []

# Process banking77
b77_names = banking['train'].features['label'].names
for split in ("train", "test"):
    for ex in banking[split]:
        src_intent = b77_names[ex["label"]]
        dm_intent  = B77_MAP.get(src_intent, "out_of_scope")
        rows.append(make_example(ex["text"], dm_intent, "banking77"))

# Process bitext
for ex in bitext['train']:
    src_intent = ex["intent"]
    dm_intent  = BITEXT_MAP.get(src_intent, "out_of_scope")
    rows.append(make_example(ex["instruction"], dm_intent, "bitext"))

df = pd.DataFrame(rows)
print(f"Total rows          : {len(df):,}")
print(f"Intent distribution :")
print(df["intent"].value_counts().to_string())


In [ ]:
# Exact-duplicate removal (same text)
before = len(df)
df = df.drop_duplicates(subset=["text"])
print(f"Exact duplicates removed: {before - len(df):,}  ({before:,} → {len(df):,})")

# Normalise text: strip, collapse whitespace
df["text"] = df["text"].str.strip().str.replace(r"\s+", " ", regex=True)
print(f"Final unified dataset: {len(df):,} examples")


## Step 6 — Carve Out the Gold Set (FROZEN)

The gold set is stratified 10% of the full dataset, carved out FIRST.
It is **never** used for training, validation, or any tuning decision.
It exists only for final model evaluation.


In [ ]:
from sklearn.model_selection import train_test_split

# Stratify on intent to ensure every class appears in gold
working_df, gold_df = train_test_split(
    df, test_size=0.10, random_state=SEED, stratify=df["intent"]
)

print(f"Gold set   : {len(gold_df):,} examples")
print(f"Working set: {len(working_df):,} examples")
print()
print("Gold intent distribution:")
print(gold_df["intent"].value_counts().to_string())


In [ ]:
# Save gold set — treat as immutable
gold_path = DATA_GOLD / "gold.jsonl"
gold_df.to_json(gold_path, orient="records", lines=True)
import os
os.chmod(gold_path, 0o444)   # read-only
print(f"Gold set saved (read-only): {gold_path}")
print(f"Size: {gold_path.stat().st_size / 1024:.1f} KB")
print()
print("IMPORTANT: data/gold/ is FROZEN. Never train on this data.")
print("Never tune hyperparameters against it.")
print("Use it only for the final model evaluation report.")


## Step 7 — Semantic Near-Duplicate Removal

Exact-duplicate removal catches identical strings. But users often resubmit
slightly reworded tickets. We use sentence embeddings to catch these.

This step uses MiniLM (fast, CPU-friendly) to embed all working-set texts,
then removes pairs with cosine similarity > 0.92.

> On Colab free tier this takes ~3–5 min for 36k examples. On CPU only, ~10 min.
> If it's too slow, lower the sample or skip this step (exact-dup removal is the critical one).


In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("Loading MiniLM encoder ...")
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = working_df["text"].tolist()
print(f"Encoding {len(texts):,} texts ...")
embeddings = encoder.encode(texts, batch_size=256, show_progress_bar=True,
                            convert_to_tensor=True, normalize_embeddings=True)
print(f"Embeddings shape: {embeddings.shape}")


In [ ]:
# Find near-duplicates in batches to avoid O(N^2) memory
# We use a greedy approach: keep the first of each near-dup cluster
THRESHOLD = 0.92
batch_size = 2000
keep_mask  = torch.ones(len(texts), dtype=torch.bool)

print(f"Finding near-duplicates (threshold={THRESHOLD}) ...")
for i in range(0, len(texts), batch_size):
    batch = embeddings[i:i+batch_size]
    # Compare this batch against all subsequent examples
    sims = (batch @ embeddings[i:].T)   # (batch, rest)
    # Zero out self-similarity and already-removed entries
    for bi in range(len(batch)):
        gi = i + bi
        if not keep_mask[gi]:
            continue
        row = sims[bi]
        # Mark all near-dups AFTER this index as removed
        near_dups = (row > THRESHOLD).nonzero(as_tuple=True)[0]
        for nd in near_dups:
            gnd = i + nd.item()
            if gnd != gi:
                keep_mask[gnd] = False

n_removed = (~keep_mask).sum().item()
print(f"Near-duplicates removed: {n_removed:,}")
working_df = working_df[keep_mask.cpu().numpy()].reset_index(drop=True)
print(f"Working set after dedup: {len(working_df):,}")


## Step 8 — Stratified Train / Val / Test Split

In [ ]:
# 80 / 10 / 10 stratified on intent
train_df, temp_df = train_test_split(
    working_df, test_size=0.20, random_state=SEED, stratify=working_df["intent"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["intent"]
)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")
print(f"Gold  : {len(gold_df):,}  (frozen)")


In [ ]:
# Verify class distributions are balanced across splits
splits = {"train": train_df, "val": val_df, "test": test_df}
dist_table = {}
for name, sdf in splits.items():
    dist_table[name] = sdf["intent"].value_counts(normalize=True).round(3)

dist_df = pd.DataFrame(dist_table).fillna(0)
print("Intent distribution (fraction) across splits:")
print(dist_df.to_string())


## Step 9 — Save Splits & Datasheet

In [ ]:
for name, sdf in splits.items():
    path = DATA_PROC / f"{name}.jsonl"
    sdf.to_json(path, orient="records", lines=True)
    print(f"Saved {name:5s}: {path}  ({path.stat().st_size / 1024:.1f} KB)")


In [ ]:
# Datasheet
print("=" * 65)
print("DESKMATE ENCODER DATASET — DATASHEET")
print("=" * 65)
print(f"  Created      : 2024")
print(f"  Sources      : PolyAI/banking77, bitext/Bitext-customer-support")
print(f"  Total rows   : {len(df):,}")
print(f"  After dedup  : {len(working_df) + len(gold_df):,}")
print()
print(f"  Gold (frozen): {len(gold_df):,}")
print(f"  Train        : {len(train_df):,}")
print(f"  Val          : {len(val_df):,}")
print(f"  Test         : {len(test_df):,}")
print()
print(f"  Intents      : {len(INTENTS)} classes (incl. out_of_scope)")
print(f"  Categories   : account, billing, data, product, technical, out_of_scope")
print(f"  Priority     : high, medium, low")
print()
print("  Priority distribution (train):")
for pri, cnt in train_df["priority"].value_counts().items():
    print(f"    {pri:<8}: {cnt:>5} ({cnt/len(train_df)*100:.1f}%)")
print()
print("  Leakage controls:")
print("    1. Exact-duplicate removal on text field")
print("    2. Semantic near-dup removal (MiniLM, sim > 0.92)")
print("    3. Gold set carved BEFORE any split — no cross-boundary dedup")
print("    4. Stratified split to preserve class ratios")
print("    5. Input: raw text only — no metadata features")


## Step 10 — Optional: Push to HuggingFace Hub

Skip this cell if you don't have an HF write token set up yet.


In [ ]:
PUSH_TO_HUB = False   # set True to push

if PUSH_TO_HUB:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        import os
        HF_TOKEN = os.getenv("HF_TOKEN", "")

    login(token=HF_TOKEN)

    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df),
        "val":   Dataset.from_pandas(val_df),
        "test":  Dataset.from_pandas(test_df),
        "gold":  Dataset.from_pandas(gold_df),
    })

    HF_USERNAME = "YOUR_HF_USERNAME"   # replace
    ds.push_to_hub(f"{HF_USERNAME}/deskmate-encoder-data", private=True)
    print("Dataset pushed to HF Hub.")
else:
    print("Skipped Hub push (PUSH_TO_HUB=False).")
    print("To push, set PUSH_TO_HUB=True and set HF_TOKEN secret.")


## Checkpoint

> **Name two leakage traps in support data and how you avoided them; justify your out-of-scope class.**


In [ ]:
answer = """
[Write your answer here]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Frozen gold set | `data/gold/gold.jsonl` (read-only) |
| Training data | `data/processed/train.jsonl` |
| Validation data | `data/processed/val.jsonl` |
| Test data | `data/processed/test.jsonl` |

**Next:** Module 2.2 — Synthetic data generation.
The public datasets give us general support intent coverage but lack DeskMate-specific
signals: your product names, version strings, error codes, and extraction spans.
Module 2.2 generates those using a teacher LLM.
